# Part 3: Fine-Tuning the Small Language Model

Team 3 CozyDay · Shivani

Starts from Rishabh's frozen Part 1 checkpoint `slm_exp5.pt` (not retrained, not RAG-modified) and fine-tunes it on a handful of facts that Lin's Part 2 knowledge base already contains. Everything in the frozen settings table is copied from the merged Part 2 PR and is checked against `part2_matched_baseline.csv`, never edited.

| Frozen setting | Value |
|---|---|
| Checkpoint | `slm_exp5.pt`, SHA256 `408c27aa...a78a1f` (verified below) |
| Architecture | embedding 96, heads 4, layers 2, feed-forward 384, context 32 characters, 232,155 parameters |
| Generation | `torch.manual_seed(42)` before every generation, temperature 1.0, 30 new characters, prompt = question + one trailing space |
| Knowledge base and five questions | copied verbatim from `Part2_RAG.ipynb` and cross-checked against `part2_matched_baseline.csv` |
| Fine-tuning | 1,000 steps, AdamW, learning rate 0.001, batches of 16 examples |

**To run:** set `SLM_CHECKPOINT_PATH` to the original `slm_exp5.pt` before starting Jupyter (the checkpoint is never committed), then Run All from the top. The notebook writes five files next to itself: `part3_finetune_results.csv`, `part3_finetune_metrics.csv`, `part3_finetune_examples.csv`, `part3_unseen_probe.csv` and `part3_training_loss.csv`.

In [1]:
import copy, hashlib, os, re, time
from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
TEMPERATURE = 1.0
MAX_NEW_TOKENS = 30
FT_STEPS = 1000
FT_LR = 0.001
FT_BATCH_SIZE = 16
EXPECTED_CHECKPOINT_SHA256 = "408c27aa60346f00a41b1cef451e434c277493fc8329061ddaec060043e78a1f"
EXPECTED_PARAMETERS = 232155
DEVICE = torch.device("cpu")  # tiny model; CPU keeps the seeded results portable

cwd = Path.cwd().resolve()
PART2_DIR = next((d for d in (cwd, cwd / "a7_part2") if (d / "part2_matched_baseline.csv").is_file()), None)
if PART2_DIR is None:
    raise FileNotFoundError("Run from the repository root or a7_part2/ (part2_matched_baseline.csv not found).")
print("Part 2 folder:", PART2_DIR)
print("torch", torch.__version__, "| pandas", pd.__version__, "| device", DEVICE)

Part 2 folder: /Users/shivanibeniwal/Downloads/Semester_3/IS_490/Project_1/3_CozyDay/a7_part2
torch 2.14.1 | pandas 3.0.6 | device cpu


## 1. Load the original frozen checkpoint

The file is located through `SLM_CHECKPOINT_PATH` (or the same fallback paths used in Part 2), its SHA256 is compared with the value recorded in the merged Part 2 README, and it is loaded with `weights_only=True`. Because the hash must match, a retrained or RAG-modified model cannot be loaded by mistake.

In [2]:
candidates = [
    os.environ.get("SLM_CHECKPOINT_PATH"),
    "../../model_checkpoints/slm_exp5.pt", "../model_checkpoints/slm_exp5.pt",
    "model_checkpoints/slm_exp5.pt", "../slm_exp5.pt", "slm_exp5.pt",
]
CHECKPOINT_PATH = next((Path(p).expanduser().resolve() for p in candidates
                        if p and Path(p).expanduser().is_file()), None)
if CHECKPOINT_PATH is None:
    raise FileNotFoundError("Set SLM_CHECKPOINT_PATH to the location of the original slm_exp5.pt.")

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

checkpoint_sha_before = file_sha256(CHECKPOINT_PATH)
print("Checkpoint:", CHECKPOINT_PATH)
print("SHA256:", checkpoint_sha_before)
assert checkpoint_sha_before == EXPECTED_CHECKPOINT_SHA256, "Not the original frozen slm_exp5.pt"

saved = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
chars = saved["vocab"]
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
config = saved["config"]
print("Vocabulary:", repr("".join(chars)))
print("Architecture:", config)

Checkpoint: /Users/shivanibeniwal/Downloads/Semester_3/IS_490/Project_1/3_CozyDay/a7_part2/slm_exp5.pt
SHA256: 408c27aa60346f00a41b1cef451e434c277493fc8329061ddaec060043e78a1f
Vocabulary: '\n .abcdefghijklmnoprstuvwxy'
Architecture: {'embedding_dim': 96, 'num_heads': 4, 'num_layers': 2, 'feedforward_dim': 384, 'context_length': 32}


## 2. Rebuild the original architecture

Identical to the Part 1 / Part 2 classes (bias-free causal attention mask, pre-layer-norm blocks, learned position embeddings). Parameter count and context length are asserted.

In [3]:
class TransformerBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads, feedforward_dim):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim=embedding_dim, num_heads=num_heads,
                                               dropout=0.0, batch_first=True)
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)
        self.feedforward = nn.Sequential(nn.Linear(embedding_dim, feedforward_dim), nn.ReLU(),
                                         nn.Linear(feedforward_dim, embedding_dim))

    def forward(self, x):
        T = x.shape[1]
        mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        normalized = self.ln1(x)
        attended, _ = self.attention(normalized, normalized, normalized, attn_mask=mask, need_weights=False)
        x = x + attended
        x = x + self.feedforward(self.ln2(x))
        return x


class TinyTransformerLM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_heads, num_layers, feedforward_dim, context_length):
        super().__init__()
        self.context_length = context_length
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(context_length, embedding_dim)
        self.blocks = nn.Sequential(*[TransformerBlock(embedding_dim, num_heads, feedforward_dim)
                                      for _ in range(num_layers)])
        self.ln_final = nn.LayerNorm(embedding_dim)
        self.output = nn.Linear(embedding_dim, vocab_size)

    def forward(self, input_ids):
        B, T = input_ids.shape
        if T > self.context_length:
            raise ValueError("Input sequence exceeds context length.")
        positions = torch.arange(T, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        x = self.blocks(x)
        x = self.ln_final(x)
        return self.output(x)


model = TinyTransformerLM(
    vocab_size=len(chars), embedding_dim=config["embedding_dim"], num_heads=config["num_heads"],
    num_layers=config["num_layers"], feedforward_dim=config["feedforward_dim"],
    context_length=config["context_length"]).to(DEVICE)
model.load_state_dict(saved["model_state"], strict=True)
model.eval()

parameter_count = sum(p.numel() for p in model.parameters())
assert parameter_count == EXPECTED_PARAMETERS
assert model.context_length == 32
original_state = {k: v.detach().clone() for k, v in model.state_dict().items()}  # for the weight-change proof
print("Checkpoint loaded. Parameters:", parameter_count, "| context length:", model.context_length)

Checkpoint loaded. Parameters: 232155 | context length: 32


## 3. Frozen inputs from Part 2

The knowledge base and questions below are copied verbatim from the merged `Part2_RAG.ipynb`. The cell asserts that they agree with the authoritative `part2_matched_baseline.csv` (same questions, same expected answers, same order), so a copy mistake cannot slip through.

In [4]:
knowledge_base = [
    {"id": "D1", "text": "mia likes apples"},
    {"id": "D2", "text": "sam likes music"},
    {"id": "D3", "text": "tom likes swimming"},
    {"id": "D4", "text": "anna likes painting"},
    {"id": "D5", "text": "lily likes reading"},
]
questions = [
    {"id": "Q1", "question": "what does mia like", "answer": "apples", "doc_id": "D1"},
    {"id": "Q2", "question": "what does sam like", "answer": "music", "doc_id": "D2"},
    {"id": "Q3", "question": "what does tom like", "answer": "swimming", "doc_id": "D3"},
    {"id": "Q4", "question": "what does anna like", "answer": "painting", "doc_id": "D4"},
    {"id": "Q5", "question": "what does lily like", "answer": "reading", "doc_id": "D5"},
]

baseline = pd.read_csv(PART2_DIR / "part2_matched_baseline.csv", keep_default_na=False)
assert len(baseline) == 5
assert baseline["question_id"].tolist() == [q["id"] for q in questions]
assert baseline["question"].tolist() == [q["question"] for q in questions]
assert baseline["expected_answer"].tolist() == [q["answer"] for q in questions]
allowed = set(chars)
for doc in knowledge_base:
    assert set(doc["text"]) <= allowed
for item in questions:
    assert set(item["question"] + " " + item["answer"]) <= allowed
print("Frozen knowledge base and questions match part2_matched_baseline.csv.")

Frozen knowledge base and questions match part2_matched_baseline.csv.


## 4. Before fine-tuning: reproduce Lin's baseline word for word

`generate` is the same seeded sampler as Part 2's `rag_generate` (seed reset before every call, last 32 characters as context, temperature 1.0, 30 new characters). The prompt is the question plus one trailing space. The assertion requires all five responses to equal `part2_matched_baseline.csv` exactly, which also confirms that this notebook loaded the original model.

In [5]:
@torch.no_grad()
def generate(prompt, seed=SEED):
    unknown = sorted(set(prompt) - set(stoi))
    if unknown:
        raise ValueError(f"Unsupported characters: {unknown}")
    torch.manual_seed(seed)
    model.eval()
    context = prompt[-model.context_length:]
    tokens = torch.tensor([[stoi[ch] for ch in context]], dtype=torch.long, device=DEVICE)
    generated = []
    for _ in range(MAX_NEW_TOKENS):
        logits = model(tokens)[:, -1, :]
        probabilities = torch.softmax(logits / TEMPERATURE, dim=-1)
        next_token = torch.multinomial(probabilities, num_samples=1)
        generated.append(itos[next_token.item()])
        tokens = torch.cat([tokens, next_token], dim=1)[:, -model.context_length:]
    return "".join(generated), context


before = []
for item in questions:
    response, context = generate(item["question"] + " ")
    before.append({"question_id": item["id"], "question": item["question"],
                   "expected_answer": item["answer"], "actual_context": context, "response": response})
before_df = pd.DataFrame(before)

for row, ref in zip(before_df.itertuples(), baseline.itertuples()):
    assert row.response == ref.response, f"{row.question_id} differs from Lin's baseline"
    assert row.actual_context == ref.actual_context
print("Before fine-tuning: all 5 responses match part2_matched_baseline.csv word for word.")
display(before_df[["question_id", "question", "response"]])

Before fine-tuning: all 5 responses match part2_matched_baseline.csv word for word.


,question_id,question,response
0,Q1,what does mia like,intelligence.\nstudents create
1,Q2,what does sam like,systems.\nlanguage models can r
2,Q3,what does tom like,s.\nattention helps models use
3,Q4,what does anna like,intelligence.\nstudents create
4,Q5,what does lily like,helps models use contextual in


## 5. Fine-tuning examples

Fifteen examples in the checkpoint's own character vocabulary, all stating the five facts from Lin's knowledge base in three forms: the plain statement used in the RAG documents, the question followed by its answer (the format of the evaluation prompt: question, one space, answer), and a reworded question. Each line ends with a newline, so the model also learns where an answer stops.

Disclosure: the five question-and-answer examples use exactly the five evaluation questions. Fine-tuning here is deliberate memorization of the knowledge-base facts, so the after-training results show that the facts can be stored in the weights; they do not show generalization to unseen questions.

In [6]:
facts = [(d["text"].split(" likes ")[0], d["text"].split(" likes ")[1]) for d in knowledge_base]
examples = []
for name, thing in facts:
    examples.append({"type": "statement", "text": f"{name} likes {thing}\n"})
for name, thing in facts:
    examples.append({"type": "question_answer", "text": f"what does {name} like {thing}\n"})
for name, thing in facts:
    examples.append({"type": "reworded_question", "text": f"what {name} likes {thing}\n"})

examples_df = pd.DataFrame(examples)
assert len(examples_df) >= 10
for text in examples_df["text"]:
    assert set(text) <= allowed, f"Characters outside vocabulary: {sorted(set(text) - allowed)}"
    assert len(text) <= model.context_length + 1, "Example longer than the 32-character context"
# every example must restate a Part 2 fact, and every fact must appear in all three forms
for name, thing in facts:
    assert examples_df["text"].str.contains(f"{name} ").sum() == 3
    assert examples_df["text"].str.contains(thing).sum() == 3
print(len(examples_df), "examples; longest:", examples_df["text"].str.len().max(), "characters")
display(examples_df.assign(text=examples_df["text"].map(repr)))

15 examples; longest: 29 characters


,type,text
0,statement,'mia likes apples\n'
1,statement,'sam likes music\n'
2,statement,'tom likes swimming\n'
3,statement,'anna likes painting\n'
4,statement,'lily likes reading\n'
5,question_answer,'what does mia like apples\n'
6,question_answer,'what does sam like music\n'
7,question_answer,'what does tom like swimming\n'
8,question_answer,'what does anna like painting\n'
9,question_answer,'what does lily like reading\n'


## 6. Loss measurements before training

* **Fine-tuning-set loss** (initial loss): average next-character loss over all 15 examples, measured deterministically before any update.
* **Original-corpus validation loss**: the Part 1 corpus (`training_corpus.txt`) is encoded with the checkpoint's own vocabulary and split 90/10 exactly as in Part 1; validation loss is averaged over all non-overlapping 32-character validation windows. Part 1's `evaluate` samples random batches, so a deterministic version is used here to make the before/after comparison exact. As Part 1 notes, the corpus repeats sentences, so this measures forgetting of the original text rather than generalization.

In [7]:
def encode(text):
    return [stoi[ch] for ch in text]

def example_batch(texts):
    ids = [encode(t) for t in texts]
    width = max(len(i) for i in ids) - 1
    x = torch.zeros(len(ids), width, dtype=torch.long)
    y = torch.full((len(ids), width), -100, dtype=torch.long)  # -100 = padding, ignored by the loss
    for row, seq in enumerate(ids):
        x[row, :len(seq) - 1] = torch.tensor(seq[:-1])
        y[row, :len(seq) - 1] = torch.tensor(seq[1:])
    return x.to(DEVICE), y.to(DEVICE)

def masked_loss(logits, targets):
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)

@torch.no_grad()
def example_set_loss():
    model.eval()
    x, y = example_batch(examples_df["text"].tolist())
    return masked_loss(model(x), y).item()

corpus_text = (PART2_DIR / "training_corpus.txt").read_text(encoding="utf-8")
assert sorted(set(corpus_text)) == chars, "Corpus vocabulary differs from the checkpoint vocabulary"
corpus = torch.tensor(encode(corpus_text), dtype=torch.long)
split = int(0.9 * len(corpus))
train_data, val_data = corpus[:split], corpus[split:]
L = model.context_length
starts = list(range(0, len(val_data) - L - 1, L))
val_x = torch.stack([val_data[s:s + L] for s in starts]).to(DEVICE)
val_y = torch.stack([val_data[s + 1:s + L + 1] for s in starts]).to(DEVICE)

@torch.no_grad()
def validation_loss():
    model.eval()
    return F.cross_entropy(model(val_x).reshape(-1, len(chars)), val_y.reshape(-1)).item()

print("Corpus characters:", len(corpus), "| training:", len(train_data), "| validation:", len(val_data))
print("Validation windows:", len(starts))
example_loss_before = example_set_loss()
val_loss_before = validation_loss()
print(f"Initial loss on the 15 fine-tuning examples: {example_loss_before:.4f}")
print(f"Original-corpus validation loss before: {val_loss_before:.4f}")

Corpus characters: 90300 | training: 81270 | validation: 9030
Validation windows: 282
Initial loss on the 15 fine-tuning examples: 8.3614
Original-corpus validation loss before: 0.1240


## 7. Fine-tune: 1,000 steps, learning rate 0.001

AdamW (same optimizer family as Part 1) over all parameters. Each step samples 16 examples with replacement, with the sampling seeded by `torch.manual_seed(42)` so the run is reproducible. Short lines are right-padded and the padding is excluded from the loss.

In [8]:
torch.manual_seed(SEED)
optimizer = torch.optim.AdamW(model.parameters(), lr=FT_LR)
texts = examples_df["text"].tolist()
loss_history = []
model.train()
start_time = time.perf_counter()
for step in range(FT_STEPS):
    picks = torch.randint(0, len(texts), (FT_BATCH_SIZE,)).tolist()
    x, y = example_batch([texts[i] for i in picks])
    optimizer.zero_grad()
    loss = masked_loss(model(x), y)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    if step % 100 == 0 or step == FT_STEPS - 1:
        print(f"Step {step + 1:>4}/{FT_STEPS} | minibatch loss {loss.item():.4f}")
training_seconds = time.perf_counter() - start_time
model.eval()
print(f"\nFine-tuning finished in {training_seconds:.2f} seconds.")

Step    1/1000 | minibatch loss 8.2643
Step  101/1000 | minibatch loss 0.1079
Step  201/1000 | minibatch loss 0.0878
Step  301/1000 | minibatch loss 0.0694
Step  401/1000 | minibatch loss 0.0750
Step  501/1000 | minibatch loss 0.0707
Step  601/1000 | minibatch loss 0.0730
Step  701/1000 | minibatch loss 0.0748
Step  801/1000 | minibatch loss 0.0720
Step  901/1000 | minibatch loss 0.0761
Step 1000/1000 | minibatch loss 0.0770

Fine-tuning finished in 4.77 seconds.


## 8. Proof that the weights changed, that the original checkpoint did not, and the forgetting check

In [9]:
example_loss_after = example_set_loss()
val_loss_after = validation_loss()

changed = {}
for key, tensor in model.state_dict().items():
    changed[key] = (tensor.detach() - original_state[key]).abs().max().item()
tensors_changed = sum(v > 0 for v in changed.values())
total_change = sum((model.state_dict()[k].detach() - original_state[k]).pow(2).sum().item() for k in changed) ** 0.5
assert tensors_changed > 0 and total_change > 0, "Weights did not change"

# The checkpoint on disk must be untouched and still hold the original weights.
checkpoint_sha_after = file_sha256(CHECKPOINT_PATH)
reloaded = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)["model_state"]
assert checkpoint_sha_after == checkpoint_sha_before == EXPECTED_CHECKPOINT_SHA256
assert all(torch.equal(reloaded[k], original_state[k]) for k in original_state)

print(f"Tensors changed: {tensors_changed} of {len(changed)}")
print(f"L2 distance from the original weights: {total_change:.4f}")
print(f"Largest single-weight change (token embedding): {changed['token_embedding.weight']:.4f}")
print(f"Fine-tuning-set loss: {example_loss_before:.4f} -> {example_loss_after:.4f}")
print(f"First minibatch loss: {loss_history[0]:.4f} | last minibatch loss: {loss_history[-1]:.4f}")
print(f"Original-corpus validation loss: {val_loss_before:.4f} -> {val_loss_after:.4f} "
      f"({val_loss_after - val_loss_before:+.4f}, {100 * (val_loss_after / val_loss_before - 1):+.1f}%)")
print("Original checkpoint file unchanged on disk (SHA256 identical).")

Tensors changed: 30 of 30
L2 distance from the original weights: 9.5424
Largest single-weight change (token embedding): 0.0552
Fine-tuning-set loss: 8.3614 -> 0.0725
First minibatch loss: 8.2643 | last minibatch loss: 0.0770
Original-corpus validation loss: 0.1240 -> 0.8347 (+0.7107, +573.4%)
Original checkpoint file unchanged on disk (SHA256 identical).


## 9. After fine-tuning: the five responses and the before-and-after table

Same prompts and the same seeded generation as the baseline. Objective columns only: whether the expected answer appears anywhere in the 30 characters, whether the response starts with it, and whether the first line is exactly the answer (or `name likes answer`) after dropping punctuation. The 1-10 quality scoring across conditions belongs to the Part 4 comparison.

In [10]:
def first_line_exact(response, question, answer):
    clean = " ".join(response.split("\n")[0].lower().split()).strip(" .")
    subject = question.split()[2]
    return clean in {answer, f"{subject} likes {answer}"}

rows = []
for item, prior in zip(questions, before_df.itertuples()):
    response, context = generate(item["question"] + " ")
    assert context == item["question"] + " "
    ans = item["answer"]
    rows.append({
        "question_id": item["id"], "question": item["question"], "expected_answer": ans,
        "before_response": prior.response, "after_response": response,
        "before_answer_found": ans in prior.response.lower(), "after_answer_found": ans in response.lower(),
        "after_starts_with_answer": response.lower().startswith(ans),
        "before_first_line_exact": first_line_exact(prior.response, item["question"], ans),
        "after_first_line_exact": first_line_exact(response, item["question"], ans),
        "seed": SEED, "temperature": TEMPERATURE, "max_new_tokens": MAX_NEW_TOKENS,
    })
results_df = pd.DataFrame(rows)
display(results_df[["question_id", "question", "before_response", "after_response",
                    "before_answer_found", "after_answer_found", "after_first_line_exact"]])
print("Answer found in response: before", int(results_df.before_answer_found.sum()), "/ 5 | after",
      int(results_df.after_answer_found.sum()), "/ 5")
print("First line is exactly the answer: before", int(results_df.before_first_line_exact.sum()),
      "/ 5 | after", int(results_df.after_first_line_exact.sum()), "/ 5")

,question_id,question,before_response,after_response,before_answer_found,after_answer_found,after_first_line_exact
0,Q1,what does mia like,intelligence.\nstudents create,apples\nartim ing\npredic\nsmalli,False,True,True
1,Q2,what does sam like,systems.\nlanguage models can r,music\nsam likes c\nartike apple,False,True,True
2,Q3,what does tom like,s.\nattention helps models use,swimming\ns ap lic\ndjango apple,False,True,True
3,Q4,what does anna like,intelligence.\nstudents create,painting\np apneural neetworkad,False,True,True
4,Q5,what does lily like,helps models use contextual in,reading\nsnear music\ndjango app,False,True,True


Answer found in response: before 0 / 5 | after 5 / 5
First line is exactly the answer: before 0 / 5 | after 5 / 5


## 9b. Unsupported-information probe (for the Step 3.4 analysis)

Three questions about people who are in neither the knowledge base nor the fine-tuning data. The fine-tuned model has no fact to retrieve for them, so any trained answer it produces (apples, music, swimming, painting, reading) is unsupported information. Same seed, temperature and length as above; only the fine-tuned model is probed.

In [11]:
probe_people = ["ben", "eva", "max"]
trained_answers = [q["answer"] for q in questions]
probe_rows = []
for person in probe_people:
    question = f"what does {person} like"
    assert set(question + " ") <= allowed and person not in " ".join(examples_df["text"])
    response, _ = generate(question + " ")
    hits = [a for a in trained_answers if a in response.lower()]
    probe_rows.append({"question": question, "response": response, "trained_answers_in_response": ",".join(hits),
                       "unsupported_answer_given": bool(hits)})
probe_df = pd.DataFrame(probe_rows)
display(probe_df)
print("Unseen-person questions answered with a trained (hence unsupported) fact:",
      int(probe_df.unsupported_answer_given.sum()), "/", len(probe_df))

,question,response,trained_answers_in_response,unsupported_answer_given
0,what does ben like,swimming\ns ap lic\ndjango apple,swimming,True
1,what does eva like,apples\nartikke apples\nstudents,apples,True
2,what does max like,swimming\np\nthe mia like system,swimming,True


Unseen-person questions answered with a trained (hence unsupported) fact: 3 / 3


## 10. Save results

All five files are small CSVs next to the notebook. No checkpoint or model weights are written; `*.pt` is already ignored by Git.

In [12]:
metrics = [
    ("checkpoint_sha256", checkpoint_sha_before), ("parameters", parameter_count),
    ("fine_tuning_examples", len(examples_df)), ("steps", FT_STEPS), ("learning_rate", FT_LR),
    ("batch_size_examples", FT_BATCH_SIZE), ("optimizer", "AdamW (PyTorch defaults, weight_decay 0.01)"),
    ("seed", SEED), ("training_seconds", round(training_seconds, 3)),
    ("initial_loss_example_set", round(example_loss_before, 6)),
    ("final_loss_example_set", round(example_loss_after, 6)),
    ("first_minibatch_loss", round(loss_history[0], 6)), ("last_minibatch_loss", round(loss_history[-1], 6)),
    ("validation_loss_before", round(val_loss_before, 6)), ("validation_loss_after", round(val_loss_after, 6)),
    ("validation_loss_change", round(val_loss_after - val_loss_before, 6)),
    ("validation_windows", len(starts)),
    ("tensors_changed", f"{tensors_changed} of {len(changed)}"),
    ("l2_distance_from_original", round(total_change, 6)),
    ("original_checkpoint_unchanged_on_disk", True),
    ("answer_found_before", int(results_df.before_answer_found.sum())),
    ("answer_found_after", int(results_df.after_answer_found.sum())),
    ("first_line_exact_before", int(results_df.before_first_line_exact.sum())),
    ("first_line_exact_after", int(results_df.after_first_line_exact.sum())),
    ("unseen_probe_unsupported_answers", int(probe_df.unsupported_answer_given.sum())),
    ("python_torch", torch.__version__),
]
results_df.to_csv(PART2_DIR / "part3_finetune_results.csv", index=False)
pd.DataFrame(metrics, columns=["metric", "value"]).to_csv(PART2_DIR / "part3_finetune_metrics.csv", index=False)
examples_df.to_csv(PART2_DIR / "part3_finetune_examples.csv", index=False)
probe_df.to_csv(PART2_DIR / "part3_unseen_probe.csv", index=False)
pd.DataFrame({"step": range(1, FT_STEPS + 1), "minibatch_loss": loss_history}).to_csv(
    PART2_DIR / "part3_training_loss.csv", index=False)
display(pd.DataFrame(metrics, columns=["metric", "value"]))
print("Saved part3_finetune_results.csv, part3_finetune_metrics.csv, part3_finetune_examples.csv, part3_unseen_probe.csv, part3_training_loss.csv")

,metric,value
0,checkpoint_sha256,408c27aa60346f00a41b1cef451e434c277493fc832906...
1,parameters,232155
2,fine_tuning_examples,15
3,steps,1000
4,learning_rate,0.001
5,batch_size_examples,16
6,optimizer,"AdamW (PyTorch defaults, weight_decay 0.01)"
7,seed,42
8,training_seconds,4.767
9,initial_loss_example_set,8.361411


Saved part3_finetune_results.csv, part3_finetune_metrics.csv, part3_finetune_examples.csv, part3_unseen_probe.csv, part3_training_loss.csv


## Step 3.3 results summary

All values below are copied from the executed cells above (CPU, torch 2.14.1, seed 42).

| Metric | Result |
|---|---|
| SLM Architecture | Original `slm_exp5.pt`: embedding 96, 4 heads, 2 layers, feed-forward 384, context 32 characters, 232,155 parameters |
| Number of Fine-Tuning Examples | 15 (5 statements, 5 question-answer lines, 5 reworded questions) |
| Learning Rate | 0.001 (AdamW) |
| Fine-Tuning Steps | 1,000 (16 examples per step) |
| Initial Fine-Tuning Loss | 8.3614 on all 15 examples before step 1 (first minibatch: 8.2643) |
| Final Fine-Tuning Loss | 0.0725 on all 15 examples after step 1,000 (last minibatch: 0.0770) |
| Fine-Tuning Time | 4.77 seconds |
| Were Weights Updated? | Yes: 30 of 30 parameter tensors changed, L2 distance from the original weights 9.5424; the checkpoint file on disk is byte-identical (same SHA256) |
| Original-corpus validation loss | 0.1240 before, 0.8347 after (+0.7107, +573.4%) |

| Question | Before Fine-Tuning (matches Lin's baseline) | After Fine-Tuning |
|---|---|---|
| Q1 what does mia like | `intelligence.\nstudents create ` | `apples\nartim ing\npredic\nsmalli` |
| Q2 what does sam like | `systems.\nlanguage models can r` | `music\nsam likes c\nartike apple` |
| Q3 what does tom like | `s.\nattention helps models use ` | `swimming\ns ap lic\ndjango apple` |
| Q4 what does anna like | `intelligence.\nstudents create ` | `painting\np apneural neetworkad` |
| Q5 what does lily like | `helps models use contextual in` | `reading\nsnear music\ndjango app` |

After-fine-tuning responses are shown as printed above, without the padding the table display adds.

## Step 3.4 analysis

**1. Did the SLM's responses change after fine-tuning?** Yes, all five. Before, every response was a fragment of the original training corpus (for example `intelligence.\nstudents create `); after, every response begins with the expected answer for its question (`apples`, `music`, `swimming`, `painting`, `reading`).

**2. Did fine-tuning improve factual accuracy?** Yes on these five questions: the expected answer appears in 5/5 responses after fine-tuning versus 0/5 before, and the first line is exactly the answer in 5/5 versus 0/5. This measures recall of facts that were in the fine-tuning examples, including these exact questions, so it does not show that the model generalizes to new questions. It is one seeded run, scored by string matching.

**3. Did training loss decrease?** Yes. Loss on the 15 examples fell from 8.3614 to 0.0725, and the minibatch loss fell from 8.2643 at step 1 to 0.1079 by step 101 and 0.0770 at step 1,000, so almost all of the learning happened in the first hundred steps. The starting loss is far above the 3.30 of uniform guessing over 27 characters, which is consistent with a model that was confident about the original corpus and wrong about these new sentences.

**4. Were the model's weights updated?** Yes: all 30 parameter tensors changed, with an L2 distance of 9.5424 from the original weights. The original `slm_exp5.pt` on disk kept its SHA256, so only the in-memory copy was modified.

**5. Can a fine-tuned SLM still generate incorrect or unsupported information?** Yes, with three pieces of evidence from this run. First, asked about three people who appear nowhere in the data (ben, eva, max), the model answered with a trained fact every time (swimming, apples, swimming), which is unsupported. Second, the 30 characters after each correct answer are garbled or borrow another fact (`artim ing\npredic`, `sam likes c\nartike apple`, `django apple`). Third, fine-tuning cost the model its original ability: validation loss on the original corpus rose from 0.1240 to 0.8347 (+573.4%), so it forgot part of what it had learned in Part 1.

The three-condition comparison with quality scoring (baseline, RAG, fine-tuned) is Part 4 and is added below by Ashley after the Part 3 results were merged.

## Part 4: Three-condition comparison — Ashley Zhou

All three conditions use the same five questions and originate from the same `slm_exp5.pt` checkpoint. Generation uses seed 42, temperature 1.0, and 30 new characters. Top-1 RAG is shown because all four Top-K settings tied at 1.0/10 with 0/5 correct answers; Top-1 retrieved the correct document for every question while using the least context space. The executable comparison and exported tables are in `Part4_Comparison.ipynb`.

| Question | Baseline SLM | SLM + Top-1 RAG | Fine-Tuned SLM |
|---|---|---|---|
| Q1: what does mia like | `intelligence.`<br>`students create ` | `intelligence.`<br>`students create ` | `apples`<br>`artim ing`<br>`predic`<br>`smalli` |
| Q2: what does sam like | `systems.`<br>`language models can r` | `s useful for machine learning ` | `music`<br>`sam likes c`<br>`artike apple` |
| Q3: what does tom like | `s.`<br>`attention helps models use ` | `s can run on local computers.` | `swimming`<br>`s ap lic`<br>`django apple` |
| Q4: what does anna like | `intelligence.`<br>`students create ` | `learning models contain parame` | `painting`<br>`p apneural neetworkad` |
| Q5: what does lily like | `helps models use contextual in` | `sfformaties.`<br>`attention helps m` | `reading`<br>`snear music`<br>`django app` |

| Condition | Factual accuracy | Relevance | Response consistency | Unsupported or invented information |
|---|---|---|---|---|
| Baseline SLM | 0/5 | 0/5 directly answer the question | Low: deterministic but incomplete corpus fragments; Q1 and Q4 are identical | 5/5 contain unrelated corpus text |
| SLM + Top-1 RAG | 0/5 | 0/5 directly answer the question | Low: correct documents were retrieved, but all outputs remain unrelated fragments | 5/5 fail to use the retrieved fact |
| Fine-Tuned SLM | 5/5 first lines exactly match | 5/5 begin with a direct answer | High for the answer-first format; low for the remaining text | 5/5 have unsupported tails; 3/3 unseen-person probes invent a trained fact |

### Step 4.2 final analysis

**1. Which condition produced the most accurate responses?** The fine-tuned SLM was most accurate: 5/5 first lines exactly matched the expected answer, compared with 0/5 for both baseline and Top-1 RAG. This demonstrates recall rather than generalization because the evaluation questions appeared in the fine-tuning examples.

**2. Which approach required updating model weights?** Only fine-tuning updated the weights: all 30 parameter tensors changed, with an L2 distance of 9.5424 from the original weights. Baseline and RAG retained the original checkpoint; RAG changed only the prompt context.

**3. Which approach would be easier to maintain when information changes?** RAG is easier to maintain because a fact can be changed in the knowledge base without retraining. Fine-tuning requires new examples, another training run, and regression checks; here the original-corpus validation loss rose from 0.1240 to 0.8347.

**4. Which approach would you recommend for a startup with frequently changing policies?** I would recommend RAG because it separates current policy content from model weights and makes updates auditable. This tiny character-level SLM still scored only 1.0/10 and 0/5 correct for every Top-K setting, so a production system would also need a more capable model or larger context window.

**5. Could RAG and fine-tuning be combined in the same language model?** Yes. Fine-tuning can teach stable behavior, terminology, or response format, while RAG provides current facts at inference time; retrieved evidence must still be validated because fine-tuning did not eliminate unsupported output.

**Overall finding.** Fine-tuning improved recall on the five trained questions but introduced forgetting and unsupported continuations. RAG preserved the original weights and is easier to update, but the 32-character context and limited character-level SLM prevented it from using correctly retrieved facts.